In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd

# Question 4

In [2]:
# parameters
g = 9.81

In [20]:
# Define Weibull parameters
shape = 2.2     # shape parameter (k)
loc = 15        # location parameter
mean_depth = 2
depth = mean_depth + 4.9


wind = { "direction": ['NNE', 'NE', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW', 'N'],
        "angle": [67.5, 90, 90, 67.5, 45, 22.5, 0, 22.5, 45],
        "fetch": [21300, 32415, 70420, 51500, 35430, 26450, 21990, 17860, 17000],
        "location": [17, 14, 16, 14, 16, 14, 15, 13, 12],
        "shape": [2.3, 2.3, 2.3, 2.3, 2.1, 2.2, 2.2, 2.4, 2.6],
        "depth": [depth, depth, depth, depth, depth, depth, depth, depth, depth],}
wind = pd.DataFrame(wind)

In [41]:
speed95 = []
for i in range(9):
    speed95.append(round(stats.weibull_min.ppf(0.95, c=wind.loc[i, 'shape'], loc=wind.loc[i, 'location'],scale=5), 2))

wind['speed95'] = speed95
display(wind)

,direction,angle,fetch,location,shape,depth,speed95,H_m0,T_p
0,NNE,67.5,21300,17,2.3,6.9,25.06,0.91,4.91
1,NE,90.0,32415,14,2.3,6.9,22.06,0.84,4.91
2,SW,90.0,70420,16,2.3,6.9,24.06,0.94,5.18
3,WSW,67.5,51500,14,2.3,6.9,22.06,0.87,5.03
4,W,45.0,35430,16,2.1,6.9,24.43,0.93,5.13
5,WNW,22.5,26450,14,2.2,6.9,22.23,0.83,4.79
6,NW,0.0,21990,15,2.2,6.9,23.23,0.84,4.75
7,NNW,22.5,17860,13,2.4,6.9,20.90,0.72,4.32
8,N,45.0,17000,12,2.6,6.9,19.63,0.67,4.15


In [42]:
H_inf = 0.14 # standard parameter
T_inf = 7.69 # standard parameter
H_m0 = []
T_p =[]

for i in range(9):
    F_tilde = (g*wind.loc[i, 'fetch'])/(wind.loc[i, 'speed95']**2)
    d_tilde = (g*wind.loc[i, 'depth'])/(wind.loc[i, 'speed95']**2)

    H_term1 = np.tanh(0.343*(d_tilde**1.14))
    H_term2 = 4.41*(10**-4)*(F_tilde**0.79)
    H_tilde = H_inf* (H_term1 * np.tanh(H_term2/H_term1))**0.572

    T_term1 = np.tanh(0.10*(d_tilde**2.01))
    T_term2 = 2.77*(10**-7)*(F_tilde**1.45)
    T_tilde = T_inf* (T_term1 * np.tanh(T_term2/T_term1))**0.187
    
    H_m0.append(round(H_tilde*(wind.loc[i, 'speed95']**2)/g,2))
    T_p.append(round(T_tilde*(wind.loc[i, 'speed95']/g),2))

wind['H_m0'] = H_m0
wind['T_p'] = T_p

display(wind)

,direction,angle,fetch,location,shape,depth,speed95,H_m0,T_p
0,NNE,67.5,21300,17,2.3,6.9,25.06,1.09,5.31
1,NE,90.0,32415,14,2.3,6.9,22.06,1.02,5.30
2,SW,90.0,70420,16,2.3,6.9,24.06,1.10,5.47
3,WSW,67.5,51500,14,2.3,6.9,22.06,1.04,5.35
4,W,45.0,35430,16,2.1,6.9,24.43,1.11,5.47
5,WNW,22.5,26450,14,2.2,6.9,22.23,1.01,5.23
6,NW,0.0,21990,15,2.2,6.9,23.23,1.02,5.19
7,NNW,22.5,17860,13,2.4,6.9,20.90,0.91,4.81
8,N,45.0,17000,12,2.6,6.9,19.63,0.85,4.64


In [6]:
wind.to_latex('wind_data.tex', index=False, float_format="%.2f", escape=False)

# Question 5

In [23]:
h = depth # water depth in front of the dike
H = 7.0 # height of the dike

In [8]:
tan_a = 0.2 # slop of the dike
H_m0 = wind.loc[wind['direction'] == 'NW', 'H_m0'].values[0] # significant wave height of west
beta = wind.loc[wind['direction'] == 'NW', 'angle'].values[0]
T_p = wind.loc[wind['direction'] == 'NW', 'T_p'].values[0] # peak period of west
L_deep = g*T_p**2/(2*np.pi)
xi_m10 = tan_a/((H_m0/L_deep)**0.5)
R_C = H - h # freeboard of the dike
gamma_b = 1.0 # influence factor for a berm
gamma_f = 1.0 # influence factor for roughness elements on the slope
gamma_runup = 1-0.0022*beta
gamma_overtop =1-0.0033*beta
gamma_nu = 1.0 # influence factor for a wall at the end of a slope

In [9]:
gamma_beta = gamma_overtop
q = (0.026/np.sqrt(tan_a)) * gamma_b * xi_m10 * np.exp(-(2.5*(R_C/(xi_m10*H_m0*gamma_b*gamma_f*gamma_beta*gamma_nu)))**1.3) * np.sqrt(g* (H_m0**3))
print(f'The overtopping discharge is {q*1000:.3f} l/m/s per meter of dike length.')

The overtopping discharge is 0.001 l/m/s per meter of dike length.


In [10]:
q=5 / 1000
RC = ((xi_m10 * H_m0 * gamma_b * gamma_f * gamma_beta * gamma_nu) / 2.5) * ((-np.log(q*np.sqrt(tan_a)/(0.026*gamma_b*xi_m10*np.sqrt(g*(H_m0**3)))))**(1/1.3))
print(f'The freeboard needed is {RC:.2f} m')


The freeboard needed is 0.61 m


# Question 7

In [11]:
gamma_w = 10030 # unit weight of the water
gamma_s = 16000 # unit weight of the submerged particle
d70 = 2.8e-4 # 70%-fractile of grain size distribution
d70_m = 2.08e-4 # Reference value of 70%-fractile of grain size dis-tribution
L = 40 + 5 # piping length
H = 5.21 # water level at the foreside of the dike
tan_theta = 1/3.3 # slope of the dike
k = 7.52e-4 # hydraulic conductivity of the auqifer
D = 6.0 # thickness of the aquifer
eta = 0.25 # Drag factor coefficient 
m_p = 1 # Model factor piping
nu = 1.33e-6 # Kinematic viscosity 
g = 9.81 # Gravitational acceleration
h_b = 0 # water level inside the dike
d = 2.5 - 0.5 # impermeable sand layer at the sand boil exit point
rho_sub = 1.25 # factor of safety



In [14]:
def Limit_state_Sellmeijer(L, H, d70_m, d70, tan_theta, k, D, eta, m_p, nu, g, h_b, d, rho_sub):
    F_R = (eta*(gamma_s/gamma_w)*tan_theta)
    F_S = ((d70_m / ((nu*k*L/g)**(1/3))) * ((d70/d70_m)**0.4))
    step_1 = ((D/L)**2.8)-1
    step_2 = (0.28/step_1)+0.04
    F_G = (0.91* (D/L)**step_2)
    H_c = m_p * F_R * F_S * F_G * L /rho_sub
    Z = H_c - (H - h_b - d*0.3)
    print(F_R, F_S, F_G)
    return Z, H_c

In [16]:
Z, H_c = Limit_state_Sellmeijer(L, H, d70_m, d70, tan_theta, k, D, eta, m_p, nu, g, h_b, d, rho_sub)
print(f'The critical hydraulic head difference is {H_c:.2f} m.')
print(f'The limit state {Z:.2f} m.')

0.1208495724946373 0.14098196221248152 1.4788638705179942
The critical hydraulic head difference is 0.91 m.
The limit state -3.70 m.


# Question 8

In [17]:
H_c = ((1/3)*L + 12)/6 +5/8.5
Z = H_c - (H - h_b - d*0.3)
print(Z)

0.4782352941176464
